In [ ]:
import os
import numpy as np
from IPython.display import clear_output
from google.colab import output
import csv

In [ ]:
# Diseños verificados de niveles para LOK usando casillas libres ('_'):
# - Las casillas con letras (L, O, K, T, A, X) se renderizan sobre FONDO_BLANCO.
# - Las casillas '_' son casillas libres de FONDO_BLANCO sin letra, para colocar Lokmons.
# - El símbolo '-' representa terreno fuera del puzzle ("vacio" / FONDO_GRIS).

niveles = [
    {
        "id": 1,
        "nombre": "Nivel 1 - Espacios Libres y LOK",
        "filas": 10,
        "columnas": 10,
        "mapa": (
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - L O K - - - -;"
            "- - - _ - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -"
        )
    },
    {
        "id": 2,
        "nombre": "Nivel 2 - Doble LOK con Espacios",
        "filas": 10,
        "columnas": 10,
        "mapa": (
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - L O K _ - - -;"
            "- - - L _ O K - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -"
        )
    },
    {
        "id": 3,
        "nombre": "Nivel 3 - El Desafío TLAK y Espacios Adyacentes",
        "filas": 10,
        "columnas": 10,
        "mapa": (
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - T L A K - - -;"
            "- - - _ _ - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -"
        )
    },
    {
        "id": 4,
        "nombre": "Nivel 3 - El Desafío TLAK y Espacios Adyacentes",
        "filas": 10,
        "columnas": 10,
        "mapa": (
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - T L O K - - -;"
            "- - - L _ _ - - - -;"
            "- - - _ - - - - - -;"
            "- - - A - - - - - -;"
            "- - - K - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -"
        )
    },
    {
        "id": 5,
        "nombre": "Nivel 3 - El Desafío TLAK y Espacios Adyacentes",
        "filas": 10,
        "columnas": 10,
        "mapa": (
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - K O X K - - -;"
            "- - - A T A O - - -;"
            "- - - X - - L - - -;"
            "- - - X X X L T - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -;"
            "- - - - - - - - - -"
        )
    }
]

def crear_archivo_niveles(ruta="niveles.csv"):
    campos = ["id", "nombre", "filas", "columnas", "mapa"]

    with open(ruta, mode="w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(niveles)

crear_archivo_niveles()

In [ ]:
class Celdas:
    RESET = "\033[0m"
    NEGRITA = "\033[1m"
    TEXTO_NEGRO = "\033[30m"

    # Colores de fondo ANSI para consola
    FONDO_GRIS = "\033[48;5;240m"   # Fuera del puzzle
    FONDO_NEGRO = "\033[40m"        # Usada / Bloqueada / Animal
    FONDO_BLANCO = "\033[107m"      # Disponible

    terrenos = {
        "vacio": f"{FONDO_GRIS}   {RESET}",
        "bloqueada": f"{FONDO_NEGRO}   {RESET}",
        "libre": f"{FONDO_BLANCO}   {RESET}",
    }

    simbolos = {
        "L": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} L {RESET}",
        "T": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} T {RESET}",
        "A": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} A {RESET}",
        "K": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} K {RESET}",
        "X": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} X {RESET}",
        "O": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} O {RESET}",
        "R": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} R {RESET}",
        "E": f"{FONDO_BLANCO}{TEXTO_NEGRO}{NEGRITA} E {RESET}",
        "ANIMAL": f"{FONDO_NEGRO}   {RESET}"
    }

    def __init__(self, terreno="vacio", contenido=None):
        self.terreno = terreno
        self.contenido = contenido

    def __str__(self) -> str:
        if self.contenido in self.simbolos:
            return self.simbolos[self.contenido]
        return self.terrenos.get(self.terreno, "❓")

In [ ]:
class Tablero:
    def __init__(self, filas=10, columnas=10):
        self.filas = filas
        self.columnas = columnas
        self.grid = np.array(
            [
                [Celdas(terreno="vacio") for _ in range(columnas)]
                for _ in range(filas)
            ],
            dtype=object,
        )

    def cargar_nivel_desde_csv(
        self, ruta_archivo="niveles.csv", nivel_id=1
    ) -> bool:
        """Carga la disposicion del tablero desde un archivo CSV usando el ID del nivel."""
        try:
            with open(
                ruta_archivo, mode="r", encoding="utf-8"
            ) as archivo_csv:
                lector = csv.DictReader(archivo_csv)

                for fila in lector:
                    if int(fila["id"]) == nivel_id:
                        # Reiniciar tablero a vacío
                        self.filas = int(fila["filas"])
                        self.columnas = int(fila["columnas"])
                        self.grid = np.array(
                            [
                                [
                                    Celdas(terreno="vacio")
                                    for _ in range(self.columnas)
                                ]
                                for _ in range(self.filas)
                            ],
                            dtype=object,
                        )

                        # Parsear el mapa codificado (filas separadas por ';')
                        filas_mapa = fila["mapa"].strip().split(";")
                        for f, f_str in enumerate(filas_mapa):
                            celdas_str = f_str.strip().split(" ")
                            for c, char in enumerate(celdas_str):
                                if char != "-":
                                    self.grid[f, c].terreno = "libre"
                                    self.grid[f, c].contenido = char
                        return True

            print(f"No se encontró el nivel {nivel_id} en el CSV.")
            return False

        except FileNotFoundError:
            print(f"Error: El archivo '{ruta_archivo}' no existe.")
            return False

    def verificar_victoria(self) -> bool:
        """Retorna True si todas las casillas del tablero no contienen letras pendientes."""
        for fil in range(self.filas):
            for col in range(self.columnas):
                celda = self.grid[fil, col]
                if celda.terreno == "libre" and celda.contenido != "ANIMAL":
                    return False
        return True

    def __str__(self) -> str:
        grid_str = (
            "    " + "  ".join([str(c) for c in range(self.columnas)]) + "\n"
        )
        for fila in range(self.filas):
            grid_str += f"{fila}  "
            for columna in range(self.columnas):
                grid_str += str(self.grid[fila, columna])
            grid_str += "\n"
        return grid_str


    def consumir_casilla(self, f: int, c: int):
        self.grid[f, c].contenido = None
        self.grid[f, c].terreno = "bloqueada"


    def es_linea_conectada(self, pos1: tuple, pos2: tuple) -> bool:
        """
        Verifica si pos1 y pos2 están en la misma línea (fila o columna)
        y si el camino entre ellas solo contiene espacio consumido ("bloqueada"),
        Lokmons, 'X', o '?' funcionando como conector.
        """
        f1, c1 = pos1
        f2, c2 = pos2

        if f1 != f2 and c1 != c2:
            return False

        paso_c = 1 if c2 > c1 else (-1 if c2 < c1 else 0)
        paso_f = 1 if f2 > f1 else (-1 if f2 < f1 else 0)

        f_actual, c_actual = f1 + paso_f, c1 + paso_c
        while (f_actual, c_actual) != (f2, c2):
            celda = self.grid[f_actual, c_actual]

            # Una casilla es transitable si está quemada/bloqueada, tiene ANIMAL,
            # o contiene un conductor 'X' o '?'
            es_pasable = (
                (celda.terreno == "bloqueada") or
                (celda.contenido in ["ANIMAL", "X", "?"])
            )
            if not es_pasable:
                return False

            f_actual += paso_f
            c_actual += paso_c

        return True

    def consumir_casilla(self, f: int, c: int):
        """
        Las casillas 'X' NO se consumen (permanecen libres y blancas).
        Las demás letras se convierten en terreno "bloqueado".
        """
        celda = self.grid[f, c]
        if celda.contenido != "X":
            celda.contenido = None
            celda.terreno = "bloqueada"

    def validar_secuencia_adyacente(self, coordenadas: list) -> bool:
        if len(coordenadas) <= 1:
            return True
        for i in range(len(coordenadas) - 1):
            if not self.es_linea_conectada(coordenadas[i], coordenadas[i + 1]):
                return False
        return True

In [ ]:
class JuegoLOKConsole:

    def __init__(self, nivel=1):
        self.nivel_actual = nivel
        self.tablero = Tablero()
        self.tablero.cargar_nivel_desde_csv("niveles.csv", nivel_id=nivel)

        self.estado_juego = "NORMAL"
        self.animales_pendientes = 0
        self.primer_animal_pos = None
        self.requiere_adyacencia = False
        self.mensaje_estado = f"¡Nivel {nivel} cargado! Ingresa coordenadas para formar palabras."

    def __mostrar_tablero(self):
        """Limpia la pantalla e imprime el estado actual del tablero."""
        output.clear()
        print(f"      LOK GAME - NIVEL {self.nivel_actual}")
        print(self.tablero)
        print(f"ESTADO: {self.mensaje_estado}\n")

    def procesar_normal(self, entrada: str):
        try:
            coords = [tuple(map(int, par.split(','))) for par in entrada.strip().split()]

            # 1. Validar conexión tramo a tramo (soporta giros de 90° pasando por 'X')
            if not self.tablero.validar_secuencia_adyacente(coords):
                self.mensaje_estado = "Las celdas no están conectadas en línea recta o vía 'X'/'?'."
                return

            letras = []
            casillas_a_consumir = []

            # 2. Interpretar la secuencia
            for f, c in coords:
                celda = self.tablero.grid[f, c]
                contenido = celda.contenido

                if celda.terreno == "bloqueada":
                    self.mensaje_estado = f"La casilla ({f},{c}) está desactivada."
                    return

                # La 'X' actúa como conductor dentro del camino ingresado (no suma letra ni se consume)
                if contenido == "X":
                    continue

                # Comodín '?'
                elif contenido == "?":
                    letras.append("?")
                    casillas_a_consumir.append((f, c))

                elif contenido and contenido != "ANIMAL":
                    letras.append(contenido)
                    casillas_a_consumir.append((f, c))
                else:
                    self.mensaje_estado = f"Casilla ({f},{c}) no contiene una letra o conductor válido."
                    return

            # 3. Mapeo de palabra
            palabra_bruta = "".join(letras)
            palabra = self._resolver_comodin(palabra_bruta)

            if palabra in ["LOK", "TLAK", "TA"]:
                # Consumir solo las letras normales/comodines (las 'X' se mantienen intactas)
                for f, c in casillas_a_consumir:
                    self.tablero.consumir_casilla(f, c)

                if palabra == "LOK":
                    self.animales_pendientes = 1
                    self.requiere_adyacencia = False
                    self.estado_juego = "COLOCAR_ANIMAL"
                    self.mensaje_estado = "¡LOK! Coloca 1 Lokmon (puedes usar casillas libres o casillas con 'X')."

                elif palabra == "TLAK":
                    self.animales_pendientes = 2
                    self.primer_animal_pos = None
                    self.requiere_adyacencia = True
                    self.estado_juego = "COLOCAR_ANIMAL"
                    self.mensaje_estado = "¡TLAK! Coloca el 1º Lokmon (el 2º deberá ser adyacente)."

                elif palabra == "TA":
                    self.estado_juego = "SELECCIONAR_OBJETIVO_TA"
                    self.mensaje_estado = "¡TA formado! Elige la casilla tipo para tus Lokmons."
            else:
                self.mensaje_estado = f"'{palabra_bruta}' no forma una palabra válida (LOK, TLAK, TA)."

        except Exception:
            self.mensaje_estado = "Formato inválido. Ejemplo: 3,3 3,4 3,5"

    def _resolver_comodin(self, palabra: str) -> str:
        """Resuelve el comodín '?' según encaje en LOK, TLAK o TA."""
        if len(palabra) == 3:
            # Evaluar si encaja con LOK (ej. "?OK", "L?K", "LO?")
            if all(p == c or p == "?" for p, c in zip(palabra, "LOK")):
                return "LOK"
        elif len(palabra) == 4:
            # Evaluar si encaja con TLAK (ej. "T?AK", "TLA?")
            if all(p == c or p == "?" for p, c in zip(palabra, "TLAK")):
                return "TLAK"
        elif len(palabra) == 2:
            # Evaluar si encaja con TA (ej. "?A", "T?")
            if all(p == c or p == "?" for p, c in zip(palabra, "TA")):
                return "TA"
        return palabra

    def procesar_ta_objetivo(self, entrada: str):
        try:
            f, c = map(int, entrada.strip().split(','))
            celda_objetivo = self.tablero.grid[f, c]

            if celda_objetivo.terreno != "libre" or celda_objetivo.contenido == "ANIMAL":
                self.mensaje_estado = f"Casilla ({f},{c}) no es válida."
                return

            tipo_objetivo = celda_objetivo.contenido
            colocados = 0

            for fil in range(self.tablero.filas):
                for col in range(self.tablero.columnas):
                    celda = self.tablero.grid[fil, col]
                    if celda.terreno == "libre" and celda.contenido == tipo_objetivo:
                        celda.contenido = "ANIMAL"
                        colocados += 1

            self.estado_juego = "NORMAL"
            tipo_txt = tipo_objetivo if tipo_objetivo else "espacios libres"

            if self.tablero.verificar_victoria():
                self.mensaje_estado = f"¡VICTORIA! Se colocaron {colocados} Lokmon en '{tipo_txt}'."
            else:
                self.mensaje_estado = f"¡TA aplicado! Se colocaron {colocados} Lokmon en '{tipo_txt}'."

        except Exception:
            self.mensaje_estado = "Formato inválido. Ejemplo: 5,5"

    def procesar_animal(self, entrada: str):
        try:
            f, c = map(int, entrada.strip().split(','))
            celda = self.tablero.grid[f, c]

            # REGLA: Puedes colocar un Lokmon en cualquier casilla jugable activa ("libre")
            # que no tenga ya un Lokmon previamente.
            if celda.terreno == "libre" and celda.contenido != "ANIMAL":

                # Validación de adyacencia para el 2º animal de TLAK
                if self.requiere_adyacencia and self.primer_animal_pos is not None:
                    if not self.tablero.es_linea_conectada(self.primer_animal_pos, (f, c)):
                        self.mensaje_estado = f"El 2º Lokmon debe estar en línea conectada con el 1º ({self.primer_animal_pos})."
                        return

                # Coloca el Lokmon (sobrescribe cualquier letra o contenido previo)
                celda.contenido = "ANIMAL"
                self.animales_pendientes -= 1

                if self.requiere_adyacencia and self.primer_animal_pos is None:
                    self.primer_animal_pos = (f, c)

                if self.animales_pendientes > 0:
                    self.mensaje_estado = f"1º Lokmon colocado en ({f},{c}). Pon el 2º adyacente."
                else:
                    self.estado_juego = "NORMAL"
                    self.requiere_adyacencia = False
                    self.primer_animal_pos = None

                    if self.tablero.verificar_victoria():
                        self.mensaje_estado = "¡VICTORIA! Tablero completado con éxito."
                    else:
                        self.mensaje_estado = f"Lokmon puesto en ({f},{c})."
            else:
                self.mensaje_estado = "Casilla no válida (está desactivada o ya tiene un Lokmon)."

        except Exception:
            self.mensaje_estado = "Formato inválido. Ejemplo: 4,4"

    def jugar(self) -> bool:
        """Bucle principal (main_loop). Retorna True si gana el nivel, False si sale."""
        while True:
            # 1. Mostrar tablero
            self.__mostrar_tablero()

            # 2. Condicional de salida (Victoria)
            if self.tablero.verificar_victoria():
                print(f"Has completado el Nivel {self.nivel_actual}.")
                input("\nPresiona ENTER para continuar al siguiente nivel...")
                return True

            # 3. Pedir entrada
            try:
                entrada = input("Ingresa tu jugada (o 's' / 'r'): ").strip()
            except (KeyboardInterrupt, EOFError):
                print("\nJuego cancelado.")
                return False

            # 4. Comandos del sistema
            if entrada.lower() == "s":
                print("Juego finalizado.")
                return False

            if entrada.lower() == "r":
                self.__init__(nivel=self.nivel_actual)
                continue

            if not entrada:
                continue

            # 5. Enrutamiento según el estado del juego
            if self.estado_juego == "NORMAL":
                self.procesar_normal(entrada)
            elif self.estado_juego == "SELECCIONAR_OBJETIVO_TA":
                self.procesar_ta_objetivo(entrada)
            elif self.estado_juego == "COLOCAR_ANIMAL":
                self.procesar_animal(entrada)

In [ ]:
def ejecutar_juego():
    nivel_actual = 1
    total_niveles = len(niveles)

    while nivel_actual <= total_niveles:
        juego = JuegoLOKConsole(nivel=nivel_actual)

        # juego.jugar() retorna True si el jugador ganó el nivel
        # o False si escribió 'salir' / canceló el juego.
        gano_nivel = juego.jugar()

        if gano_nivel:
            nivel_actual += 1
        else:
            print("\nHas salido del juego.")
            break

    if nivel_actual > total_niveles:
        print("\nHas completado todos los niveles de LOK.")

# Punto de entrada del programa
if __name__ == "__main__":
    ejecutar_juego()

      LOK GAME - NIVEL 1
    0  1  2  3  4  5  6  7  8  9
0                                
1                                
2                                
3            L  O  K             
4                                
5                                
6                                
7                                
8                                
9                                

ESTADO: ¡Nivel 1 cargado! Ingresa coordenadas para formar palabras.


Juego cancelado.

Has salido del juego.
Ingresa tu jugada (o 's' / 'r'): s
